# Metrics

In [ ]:
# тестовый датасет
test_dataset = [
    {
        "question": "Что такое LangGraph?",
        "expected_answer_contains": ["граф", "workflow", "агент"],
        "relevant_docs": ["doc2"],
        "should_cite": True
    },
    {
        "question": "Для чего используется RAG?",
        "expected_answer_contains": ["retrieval", "generation", "знания"],
        "relevant_docs": ["doc3"],
        "should_cite": True
    },
    {
        "question": "Привет, как дела?",
        "expected_answer_contains": None,
        "relevant_docs": [],
        "should_cite": False
    },
]

## Метрики качества retrieval

In [2]:
# Precision@k
# Показывает долю релевантных документов среди первых k найденных:

def calculate_precision_at_k(retrieved_docs: list, relevant_docs: list, k: int) -> float:
    retrieved_top_k = retrieved_docs[:k]
    relevant_found = sum(1 for doc in relevant_docs if doc in retrieved_top_k)
    return relevant_found / k if k > 0 else 0.0

In [3]:
# Recall@k
# Показывает какой процент всех релевантных документов был найден:

def calculate_recall_at_k(retrieved_docs: list, relevant_docs: list, k: int) -> float:
    retrieved_top_k = retrieved_docs[:k]
    relevant_found = sum(1 for doc in relevant_docs if doc in retrieved_top_k)
    return relevant_found / len(relevant_docs) if relevant_docs else 0.0

                  


In [4]:
# MRR (Mean Reciprocal Rank)
# Насколько высоко стоит первый релевантный документ:

def calculate_mrr(test_results: list) -> float:
    reciprocal_ranks = []
    for r in test_results:
        retrieved = r.get("retrieved_docs", [])
        relevant = set(r.get("relevant_docs", []))
        rank = next((i+1 for i, doc in enumerate(retrieved) if doc in relevant), None)
        reciprocal_ranks.append(1.0/rank if rank else 0.0)
    return sum(reciprocal_ranks)/len(test_results) if test_results else 0.0

## Метрики качества generation

Основные показатели качества ответов:
- Faithfulness — факты проверяемы и подтверждены источниками
- Answer Relevance — отвечает ли ответ на заданный вопрос
- Correctness — соответствует ли ответ эталонным данным
- Citation Rate — доля утверждений с корректными ссылками на источники

In [ ]:
# Простая оценка через LLM
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def evaluate_qa_with_llm(query: str, prediction: str, reference: str) -> dict:
    prompt = f"""Ты эксперт по оценке качества ответов.

Вопрос: {query}
Ожидаемый ответ: {reference}
Полученный ответ: {prediction}

Оцени, насколько полученный ответ соответствует ожидаемому по смыслу.
Ответь ТОЛЬКО одним словом: CORRECT или INCORRECT"""

    response = llm.invoke(prompt)
    verdict = response.content.strip().upper()
    score = 1.0 if verdict == "CORRECT" else 0.0
    
    return score, verdict

In [ ]:
# Использование RAGAS
# RAGAS — стандарт индустрии для оценки RAG-систем с проверенными метриками:

from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
from datasets import Dataset

data = {
    "user_input": ["Когда основана компания?"],
    "response": ["Компания основана в 1999 году."],
    "retrieved_contexts": [["Компания основана в 1999 году основателями: Иван Иванов и Мария Петрова."]],
    "reference": ["1999 год."]
}

dataset = Dataset.from_dict(data)

result = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),           # Соответствие контексту (нет галлюцинаций)
        AnswerRelevancy(),        # Релевантность ответа вопросу
        ContextRecall()           # Полнота извлеченного контекста
    ]
)

print(f"Faithfulness: {result['faithfulness']:.3f}")
print(f"Answer Relevancy: {result['answer_relevancy']:.3f}")
print(f"Context Recall: {result['context_recall']:.3f}")

In [ ]:
# Автоматическая оценка на тестовом датасете
def evaluate_agent(app, test_dataset):
    results = []
    print("Запуск автоматической оценки...\n")
    
    for i, test_case in enumerate(test_dataset, 1):
        query = test_case["question"]
        response = app.invoke({"messages": [HumanMessage(content=query)]})
        messages = response["messages"]
        final_answer = messages[-1].content
        
        retrieved_docs = []
        for msg in messages:
            if msg.type == "tool" and hasattr(msg, 'artifact') and msg.artifact:
                retrieved_docs = [d.metadata.get("source") for d in msg.artifact]
                break
        
        has_citations = "[источник" in final_answer.lower()
        answer_quality = evaluate_answer(final_answer, test_case.get("expected_answer_contains"))
        
        result = {
            "question": query,
            "retrieved_docs": retrieved_docs,
            "relevant_docs": test_case.get("relevant_docs", []),
            "has_citations": has_citations,
            "should_cite": test_case.get("should_cite", False),
            "answer_quality": answer_quality
        }
        results.append(result)
        
        hit = bool(set(retrieved_docs) & set(result["relevant_docs"]))
        citation_ok = has_citations == test_case.get("should_cite", False)
        status = "✅" if (hit or not result["relevant_docs"]) and citation_ok else "⚠️"
        
        print(f"[{i}/{len(test_dataset)}] {status} Retrieved: {retrieved_docs}, Quality: {answer_quality}")
    
    precision = sum(1 for r in results if set(r["retrieved_docs"]) & set(r["relevant_docs"])) / len(results)
    mrr = calculate_mrr(results)
    citation_rate = sum(r["has_citations"] for r in results)/len(results)
    
    print(f"\n{'='*60}")
    print("Итоговые метрики:")
    print(f"Precision:       {precision:.1%}")
    print(f"MRR:             {mrr:.3f}")
    print(f"Citation Rate:   {citation_rate:.1%}")
    print(f"{'='*60}\n")
    
    return results

def evaluate_answer(prediction: str, expected_keywords: list) -> str:
    if not expected_keywords:
        return "FLEXIBLE"
    found = sum(1 for kw in expected_keywords if kw.lower() in prediction.lower())
    return "GOOD" if found >= len(expected_keywords)/2 else "POOR"

evaluation_results = evaluate_agent(app, test_dataset)

| Метрика        | Что оценивает                        | Если низкая                                                     |
|----------------|--------------------------------------|------------------------------------------------------------------|
| Precision@k    | доля релевантных в топ-k            | много лишних документов, нужен reranker                         |
| Recall@k       | полнота покрытия                    | система пропускает нужные документы, увеличить k или улучшить embeddings |
| Faithfulness   | соответствие источникам             | галлюцинации, усилить промпт или уменьшить temperature          |
| Answer Relevance | отвечает ли на вопрос             | модель уходит в сторону, улучшить промпт                        |
| Citation Rate  | корректность ссылок                 | добавить явное требование цитирования в промпт                  |

Типичные хорошие значения:
- Faithfulness ≥ 0.85
- Answer Relevance ≥ 0.90
- Precision@k ≥ 0.70
- Citation Rate ≥ 0.60 (если требуются обязательные ссылки)